### Extracting Interactive Map from Itinerary JSON files
Using `folium` for detailed interactive maps with clickable markers and routing.

In [1]:
# !pip install folium

In [7]:
import json
import folium

# 1. Load the data
original_path = "itinerary.json"
optimized_path = "tsp_results.json"
# optimized_geo_path = "tsp_results2.json"

original_itinerary = json.load(open(original_path))
optimized_itinerary = json.load(open(optimized_path))
# optimized_geo_itinerary = json.load(open(optimized_geo_path))

# 2. Function to build the Folium Map
def create_folium_map(data, title='itinerary', hotel_location=None):
    itinerary = data.get(title, {})
    
    points = []
    # Folium supports specific colors: red, blue, green, purple, orange, darkred, lightred, beige, darkblue, darkgreen, cadetblue, darkpurple, white, pink, lightblue, lightgreen, gray, black, lightgray
    day_colors = {'Day1': 'red', 'Day2': 'green', 'Day3': 'blue'}
    
    for day_name, day_content in itinerary.items():
        if hotel_location:
            points.append({
                'lat': hotel_location['lat'],
                'lon': hotel_location['lon'],
                'name': 'Hotel',
                'day': day_name,
                'time': 'N/A',
                'color': 'orange'
            })
        for time_slot, pois in day_content.items():
            for poi in pois:
                points.append({
                    'lat': poi['location']['lat'] if title == 'itinerary' else poi['latitude'],
                    'lon': poi['location']['lon'] if title == 'itinerary' else poi['longitude'],
                    'name': poi['place'],
                    'day': day_name,
                    'time': time_slot,
                    'color': day_colors.get(day_name, 'gray')
                })

    if not points:
        print(f"No points found for '{title}'")
        return None

    unique_points = []
    seen = set()
    for p in points:
        key = (p['lat'], p['lon'], p['name'])
        if key not in seen:
            seen.add(key)
            unique_points.append(p)
            
    avg_lat = sum(p['lat'] for p in unique_points) / len(unique_points)
    avg_lon = sum(p['lon'] for p in unique_points) / len(unique_points)
    
    # Create base map
    m = folium.Map(location=[avg_lat, avg_lon], zoom_start=12, tiles='CartoDB positron')
    
    # Draw route lines
    route_coords = [[p['lat'], p['lon']] for p in points]
    folium.PolyLine(
        route_coords, 
        color="black", 
        weight=2.5, 
        opacity=0.6,
        dash_array='5, 5'
    ).add_to(m)
    
    # Add interactive markers
    for idx, p in enumerate(unique_points):
        popup_html = f"""
        <div style='min-width: 150px; font-family: sans-serif;'>
            <h4 style='margin-bottom: 5px;'>{idx + 1}. {p['name']}</h4>
            <hr style='margin: 5px 0px;'>
            <b>Day:</b> {p['day']}<br>
            <b>Time:</b> {p['time']}
        </div>
        """
        
        folium.Marker(
            location=[p['lat'], p['lon']],
            popup=folium.Popup(popup_html, max_width=300),
            tooltip=f"Click for details: {p['name']}",
            icon=folium.Icon(color=p['color'], icon='info-sign')
        ).add_to(m)
        
    return m

In [9]:
print("Generating original itinerary map...")
original_map = create_folium_map(original_itinerary, title='itinerary', hotel_location={
        "lat": 36.1699,
        "lon": -115.1398
    }) 
original_map

Generating original itinerary map...


In [13]:
print("Generating optimized itinerary map...")
# Ensure the title key matches the root JSON property inside tsp_results1.json
optimized_map = create_folium_map(optimized_itinerary, title='optimized_itinerary', hotel_location={
        "lat": 36.1699,
        "lon": -115.1398
    })
optimized_map

Generating optimized itinerary map...
